# A/B/C/D 측정 · 입력 검증 · 개발 정책 고정 · 평가

05에서 실행하고 **06에서 원시 로그 대조와 그래프를 확인**한다.

- 기본 `WORKFLOW='staged'`: 이전 단계 검증의 입력 재생 → 판단 검사 → C 개별효과 → D 개발 대조 → 정책 고정 → 별도 평가.
- `WORKFLOW='standard'`: 기존 품질 캐시 / 독립 검색 지연 / 공통 LLM 전체 시간 측정을 그대로 사용한다.

단계 검증의 문항 수는 `config/validation_study.json`의 `questions_per_task`, 일반 비교는 `config/experiment.json`에서 수정한다.
기본 단계 검증은 실제 Jev·GPT API를 호출한다. `RUN_STUDY=False`이면 실행 설정만 확인한다.
모든 새 실행은 고유 폴더를 사용하고 과거 원시 결과를 보존한다. 01/04의 공통 문서·질문·트리는 재사용한다.

In [ ]:
from pathlib import Path
from uuid import uuid4
import sys, os, json, time
from dotenv import load_dotenv
ROOT = Path.cwd()
if not (ROOT/'config/c.json').exists(): ROOT = ROOT.parent
assert (ROOT/'config/c.json').exists()
load_dotenv(ROOT/'.env', override=False, encoding='utf-8-sig')
sys.path.insert(0, str(ROOT))
# Windows Jupyter: load Pandas/PyArrow before Torch to avoid arrow.dll crashes.
import pandas as pd
# Reload dependencies first, then bind current definitions in notebook kernels.
import importlib
for module_name in ('rag_baselines','rag_baselines.data','rag_baselines.store',
                    'rag_baselines.generation','rag_baselines.jev','rag_baselines.models',
                    'rag_baselines.laya_local','rag_baselines.c_tree','rag_baselines.raw_logging',
                    'rag_baselines.c_retrieval','rag_baselines.evaluation','rag_baselines.comparison','rag_baselines.validation_study','rag_baselines.validation_execution'):
    importlib.reload(importlib.import_module(module_name))
import httpx
from rag_baselines.raw_logging import RawLog
from rag_baselines.data import read_json, read_jsonl, write_json, write_jsonl, file_sha256
from rag_baselines.models import Baselines
from rag_baselines.store import DocumentStore
from rag_baselines.c_tree import load_tree
from rag_baselines.c_retrieval import JevRetrieval, ComparativeRetrieval
from rag_baselines.jev import JevClient
from rag_baselines.laya_local import LayaDecisionClient
from rag_baselines.evaluation import (evaluate_comparative_quality, measure_retrieval, measure_system,
                                     result_table, runtime_manifest, save_run, notebook_source_sha256)
from rag_baselines.generation import build_prompt
CONFIG = read_json(ROOT/'config/baselines.json')
C_CONFIG = read_json(ROOT/'config/c.json')
D_CONFIG = read_json(ROOT/'config/d_laya.json')
TAXONOMY = read_json(ROOT/'config/c_taxonomy.json')
GENERATION = read_json(ROOT/'config/generation.json')
pd.set_option('display.max_colwidth', 100)
print('Workspace:', ROOT)
print('Keys configured:', {'Jev':bool(os.environ.get('TYPESAFE_API_KEY')), 'OpenAI':bool(os.environ.get('OPENAI_API_KEY'))})

from datetime import datetime, timezone
from rag_baselines.validation_study import run_c_development, selected_questions
from rag_baselines.validation_execution import run_comparison, freeze_d_policy


In [ ]:
WORKFLOW = 'staged'  # staged=입력 검사·개발·별도 평가, standard=기존 직접 비교
RUN_STUDY = True    # staged 실행 스위치. False이면 유료 호출·모델 추론 없음.
if WORKFLOW not in ('staged', 'standard'):
    raise ValueError('WORKFLOW must be staged or standard.')
STUDY = read_json(ROOT/'config/validation_study.json') if WORKFLOW == 'staged' else None
OUT = None
if WORKFLOW == 'staged':
    print('단계 검증 설정:', STUDY['questions_per_task'], '| 실제 실행:', RUN_STUDY)
    if RUN_STUDY:
        if not STUDY['run_dev']:
            raise ValueError('A new study needs development before freezing policies. Use 06 to read saved runs.')
        if STUDY['repeats'] != 1 or not STUDY['save_raw_logs']:
            raise ValueError('Staged validation requires one repetition and raw logging.')
        dev_questions = selected_questions(ROOT, 'dev', STUDY['questions_per_task']['dev'])
        eval_questions = selected_questions(ROOT, 'eval', STUDY['questions_per_task']['eval']) if STUDY['run_eval'] else []
        planned_llm_calls = len(dev_questions) * (len(STUDY['variants']) + 4) + len(eval_questions) * 4
        if planned_llm_calls > STUDY['max_llm_calls']:
            raise ValueError(f'Study requires {planned_llm_calls} LLM calls, above max_llm_calls.')
        if not os.environ.get('OPENAI_API_KEY') or not os.environ.get('TYPESAFE_API_KEY'):
            raise RuntimeError('Set OPENAI_API_KEY and TYPESAFE_API_KEY in the project .env before execution.')
        OUT = ROOT/'results'/('staged_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')+'_'+uuid4().hex[:8])
        OUT.mkdir(parents=True, exist_ok=False)
        write_json(OUT/'study_snapshot.json', STUDY)
        write_json(OUT/'execution_source.json', {'notebook': '05_measure_abcd.ipynb',
            'notebook_source_sha256': notebook_source_sha256(ROOT/'notebook/05_measure_abcd.ipynb')})
        print('새 실행 위치:', OUT, '| GPT 호출 상한:', planned_llm_calls)

## 1. 실제 입력 패킹과 판단 검사 · 단계 검증

입력 재생은 추론 없이 지시문·선택지·본문 잘림을 확인한다. 출처를 확인한 48개 개발 판단과 선택지 순서 검사도 저장한다.
판단 검사는 CPU에서 실행하는 정확성 진단이며, A/B/D의 GPU 지연 측정과 구분한다.

In [ ]:
if WORKFLOW == 'staged' and RUN_STUDY:
    from scripts.validate_laya_replay import validate_replay
    from scripts.check_laya_decisions import run_checks
    validate_replay(ROOT/STUDY['source_run'], OUT)
    run_checks(ROOT/STUDY['source_run'], OUT, device='cpu')
    replay = read_json(OUT/'d_input_replay_summary.json')
    if replay['accepted'] != replay['decisions'] or replay['failures']:
        raise ValueError('D input replay failed. Inspect the saved diagnostics before continuing.')
    print('입력 검사 완료:', replay['accepted'], '| 판단 결과:', OUT/'decision_validation.csv')

## 2. C 개별효과 실행과 정책 고정 · 단계 검증

baseline / top2 / top3 / 동일 충분성 입력 재사용을 각각 새로운 검색·생성으로 실행한다.
정책 순서는 질문별로 섞고 한 번에 하나의 변경만 비교한다. 개발 결과로 C 설정을 고정하며 평가 결과로 재조정하지 않는다.

In [ ]:
if WORKFLOW == 'staged' and RUN_STUDY:
    dev_summary, frozen_c = run_c_development(ROOT, OUT, STUDY)
    display(dev_summary)
    print('개발에서 고정한 C:', frozen_c['selected_arm'])

## 3. D 개발 대조와 A/B/C/D 별도 평가 · 단계 검증

A/B/D는 동일 CUDA 장치에서 측정하고 CPU 폴백·입력 잘림을 거부한다.
입력 수정 대조군과 별도 D 후보를 개발 질문에서 비교한 뒤 D 설정을 고정한다.
평가 라벨 로딩 전 정책을 기록하고 A/B는 같은 검색 후보를 사용한다. 개발·평가는 한 노트북에서 순차 실행한다.

In [ ]:
if WORKFLOW == 'staged' and RUN_STUDY:
    run_comparison(ROOT, OUT, STUDY, split='dev', include_c=False)
    run_comparison(ROOT, OUT, STUDY, split='dev', include_c=False, d_candidate=True)
    d_summary, frozen_d = freeze_d_policy(ROOT, OUT)
    display(d_summary)
    print('개발에서 고정한 D:', frozen_d['selected_arm'])
    if STUDY['run_eval']:
        display(run_comparison(ROOT, OUT, STUDY, split='eval', include_c=True))
        # Change the current pointer only after the complete evaluation succeeds.
        (ROOT/'results/current_staged_run.txt').write_text(str(OUT), encoding='utf-8')
        print('실행 완료. 06에서 원시 로그·전체 비교·개발 효과·분포·비용을 확인한다.')
    else:
        print('개발만 완료. 06의 SYSTEM_RUN에 dev/ABD_input_fix 경로를 지정하면 분석할 수 있다.')
    print('원시 로그는 각 실행 manifest의 raw_log, 실제 결과는', OUT)

## 일반 비교 · 품질 캐시 / 독립 검색 지연 / 전체 시스템

아래 셀은 `WORKFLOW='standard'`에서만 실행한다. `config/experiment.json`의 문항 수·반복·검색/선택 수를 따른다.
기본 단계 검증과 중복 실행하지 않는다.

In [ ]:
if WORKFLOW == 'standard':
    EXPERIMENT_PATH = ROOT/'config/experiment.json'
    EXPERIMENT_CONFIG = read_json(EXPERIMENT_PATH)
    EXPERIMENT_SNAPSHOT = {'path':str(EXPERIMENT_PATH.relative_to(ROOT)),
                           'sha256':file_sha256(EXPERIMENT_PATH), 'parameters':EXPERIMENT_CONFIG}
    SPLIT = EXPERIMENT_CONFIG['split']
    if SPLIT not in ('dev','eval'):
        raise ValueError('experiment.json: split must be dev or eval.')
    QUESTIONS_PER_TASK = EXPERIMENT_CONFIG['questions_per_task'][SPLIT]
    REPEATS = EXPERIMENT_CONFIG['repeats']
    CORPUS_SIZE = EXPERIMENT_CONFIG['corpus_size']
    RETRIEVE_K = EXPERIMENT_CONFIG['retrieve_k']
    EVIDENCE_K = EXPERIMENT_CONFIG['evidence_k']
    RUN_RETRIEVAL = EXPERIMENT_CONFIG['run_retrieval']
    RUN_GENERATION = EXPERIMENT_CONFIG['run_generation']
    SAVE_RAW_LOGS = EXPERIMENT_CONFIG.get('save_raw_logs',True)
    METHODS = ('A','B','C','D')
    if tuple(EXPERIMENT_CONFIG.get('methods', METHODS)) != METHODS:
        raise ValueError('This experiment compares exactly A/B/C/D; keep methods unchanged.')
    for name,value in {'questions_per_task':QUESTIONS_PER_TASK, 'repeats':REPEATS,
                       'corpus_size':CORPUS_SIZE, 'retrieve_k':RETRIEVE_K, 'evidence_k':EVIDENCE_K}.items():
        if type(value) is not int or value < 1:
            raise ValueError(f'experiment.json: {name} must be a positive integer.')
    if any(type(v) is not bool for v in (RUN_RETRIEVAL,RUN_GENERATION,SAVE_RAW_LOGS)):
        raise ValueError('experiment.json: run switches must be true or false.')
    if EVIDENCE_K > min(RETRIEVE_K,CORPUS_SIZE):
        raise ValueError('evidence_k must not exceed retrieve_k or corpus_size.')
    if RUN_GENERATION and not RUN_RETRIEVAL:
        raise ValueError('System measurement needs run_retrieval=true.')
    assert GENERATION['encoding_name'] == C_CONFIG['search']['encoding_name'] == D_CONFIG['search']['encoding_name']
    assert C_CONFIG['search']['evidence_text_tokens'] == D_CONFIG['search']['evidence_text_tokens'] == 2000
    manifest = read_json(ROOT/'data/prepared/manifest.json')
    corpus_size = CORPUS_SIZE
    if str(corpus_size) not in manifest['corpora']:
        raise ValueError('Choose a prepared corpus_size (2000, 10000, 40000).')
    entry = manifest['corpora'][str(corpus_size)]
    corpus_path = ROOT/entry['path']
    assert file_sha256(corpus_path) == entry['sha256']
    all_questions = read_jsonl(ROOT/f'data/prepared/questions_{SPLIT}.jsonl')
    questions = []
    for task in CONFIG['qa']['tasks']:
        available = [q for q in all_questions if q['task']==task]
        if QUESTIONS_PER_TASK > len(available):
            raise ValueError(f'{SPLIT}/{task}: requested {QUESTIONS_PER_TASK}, available {len(available)}.')
        questions.extend(available[:QUESTIONS_PER_TASK])
    labels = read_jsonl(ROOT/f'data/prepared/labels_{SPLIT}.jsonl')
    MAX_LLM_API_CALLS = len(questions)*len(METHODS)*REPEATS
    EXPERIMENT_SNAPSHOT['selected_question_ids'] = [q['id'] for q in questions]
    display(pd.DataFrame(questions).groupby('task').size().rename('selected_questions').to_frame())
    display(pd.DataFrame(questions).head(6))
    print('Split:',SPLIT,'| total questions:',len(questions),'| repeats:',REPEATS)
    print('Run switches:',{'retrieval':RUN_RETRIEVAL,'generation':RUN_GENERATION})
    print('Maximum GPT API calls:',MAX_LLM_API_CALLS,'| Jev calls/query:',C_CONFIG['search']['max_api_calls'],
          '| Laya local calls/query:',D_CONFIG['search']['max_decision_calls'])
    print('Pinned Jev:',C_CONFIG['model'],'| pinned Laya:',D_CONFIG['model']['repo'],D_CONFIG['model']['revision'])
    print('Common generator:',GENERATION['model'])


### 일반 비교 · 공통 트리와 입력 잘림 검증

04에서 준비한 하나의 트리를 C와 D가 그대로 공유한다. D는 `convaiinnovations/laya-multilingual`을 자동 Router 없이 고정 revision으로 직접 불러온다. 첫 실행에서만 모델 가중치가 Hugging Face 캐시에 내려받아질 수 있다. A/B/D는 같은 GPU 장치를 쓰며, D가 요청한 장치에서 CPU로 fallback하면 실행을 거부한다.

C의 탐색 정책은 `config/c.json`, D의 임시 초기 정책과 token budget은 `config/d_laya.json`에서 각각 관리한다. D에서 모델 입력 잘림이 보고되면 raw request/response에 남기고 그 결과를 정상 observation으로 받지 않는다. 짧은 입력과 긴 입력의 안정성, 임계값 보정은 실험에서 별도로 확인한다.


In [ ]:
if WORKFLOW == 'standard':
    engine = None
    jev_client = None
    jev_http = None
    laya_client = None
    audit = None
    RAW_LOG = None
    RAW_SESSION_META = {'enabled':SAVE_RAW_LOGS,'session':None,
                        'timing':'diagnostic input/output capture is included in measured wall time'}
    if RUN_RETRIEVAL:
        store = DocumentStore(corpus_path)
        tree, preparation = load_tree(ROOT,store,TAXONOMY,C_CONFIG)
        if SAVE_RAW_LOGS:
            RAW_LOG = RawLog(ROOT,manifest={'notebook':'05_measure_abcd.ipynb','experiment':EXPERIMENT_SNAPSHOT,
                'corpus_sha256':entry['sha256'],'generation':GENERATION,'c_config':C_CONFIG,'d_config':D_CONFIG,
                'shared_tree_preparation':'C Jev tree reused by C and D'},
                prices={'openai':GENERATION['prices'],'jev':C_CONFIG['prices']})
            RAW_SESSION_META['session'] = str(RAW_LOG.directory.relative_to(ROOT))
            # Labels are evaluator-only snapshots, never passed to a retriever or generator.
            for path in [EXPERIMENT_PATH,ROOT/'config/baselines.json',ROOT/'config/c.json',ROOT/'config/d_laya.json',
                         ROOT/'config/c_taxonomy.json',ROOT/'config/generation.json',corpus_path,
                         ROOT/'data/prepared/manifest.json',ROOT/'data/prepared/preparation_metrics.json',
                         ROOT/f'data/prepared/questions_{SPLIT}.jsonl',ROOT/f'data/prepared/labels_{SPLIT}.jsonl',
                         ROOT/'notebook/05_measure_abcd.ipynb',*sorted((ROOT/'rag_baselines').glob('*.py'))]:
                RAW_LOG.snapshot(path)
            with RAW_LOG.scope(phase='preparation',method='shared'):
                RAW_LOG.event('tree','reused',{'preparation':preparation,'actual_new_api_calls':0,
                                                'shared_by_methods':['C','D']})
                for name in ('tree.json','manifest.json','checkpoint.json','checkpoint_hash.json'):
                    RAW_LOG.snapshot(Path(preparation['directory'])/name)
            print('Raw logs:',RAW_LOG.directory)
        baseline = Baselines(ROOT,retrieve_k=RETRIEVE_K,evidence_k=EVIDENCE_K,raw_log=RAW_LOG)
        baseline.build_index(corpus_path)
        audit = baseline.truncation_audit(read_jsonl(ROOT/'data/prepared/questions_dev.jsonl') +
                                          read_jsonl(ROOT/'data/prepared/questions_eval.jsonl'))
        if RAW_LOG:
            RAW_LOG.result('truncation_audit',audit)
            with RAW_LOG.scope(phase='warmup',method='AB'):
                baseline.warmup(questions[0]['question'])
        else:
            baseline.warmup(questions[0]['question'])
        if RAW_LOG:
            jev_http = httpx.Client(timeout=C_CONFIG['timeout_seconds'],event_hooks=RAW_LOG.http_hooks)
        jev_client = JevClient(C_CONFIG,http_client=jev_http)
        decision = JevRetrieval(ROOT,baseline.store,tree,preparation,C_CONFIG,client=jev_client)
        laya_client = LayaDecisionClient(D_CONFIG,device=baseline.device)
        local_decision = JevRetrieval(ROOT,baseline.store,tree,preparation,D_CONFIG,client=laya_client,
                                      method='D',provider='laya')
        engine = ComparativeRetrieval(baseline,decision,local_decision)
        if RAW_LOG:
            RAW_LOG.result('runtime',runtime_manifest(ROOT,engine))
        display(pd.DataFrame(audit).T)
        display(pd.DataFrame([
            {'method':'A/B','preparation_seconds':baseline.index_info['build_seconds'],'api_cost_usd':None,
             'input_tokens':baseline.index_info['document_tokens']['nonpad'],'cache_hit':baseline.index_info['cache_hit']},
            {'method':'C+D shared tree','preparation_seconds':preparation['prepare_seconds'],
             'api_cost_usd':preparation['api_cost_usd'],'input_tokens':preparation['api_usage']['input_tokens'],
             'cache_hit':preparation['cache_hit']},
            {'method':'D local Laya model','preparation_seconds':laya_client.load_seconds,
             'api_cost_usd':0.0,'input_tokens':None,'cache_hit':None},
            {'method':'D local Laya warmup','preparation_seconds':laya_client.warmup_seconds,
             'api_cost_usd':0.0,'input_tokens':None,'cache_hit':None},
        ]))
        print('C taxonomy authoring time:',preparation['human_taxonomy_authoring_seconds'],'(unmeasured)')
        print('C and D share this Jev-prepared tree; D uses no decision API. Local GPU dollar cost is unmeasured.')
        print('D runtime:',laya_client.runtime_info())
    else:
        print('Retrieval measurement is disabled in config/experiment.json.')


### 일반 비교 · 저장된 검색 결과로 품질 평가

A/B는 dense 후보 ID·점수가 같은지 확인한다. C와 D는 같은 트리에서 각각의 판단 결과를 별도 품질 캐시에 기록한다. 첫 캐시 생성은 실제 Jev API 또는 로컬 Laya 추론을 실행한다. 캐시를 다시 쓰는 동안의 지연은 실시간 검색 지연으로 해석하지 않는다. 정답 라벨은 검색이 끝난 뒤 평가기에만 전달한다.


In [ ]:
if WORKFLOW == 'standard':
    quality = []
    quality_directory = None
    if engine:
        quality = evaluate_comparative_quality(engine,questions,labels,audit=RAW_LOG)
        for q in questions:
            pair = [r for r in quality if r['question_id']==q['id'] and r['method'] in ('A','B')]
            assert [(p['id'],p['dense_score']) for p in pair[0]['candidates']] == [(p['id'],p['dense_score']) for p in pair[1]['candidates']]
        for r in quality:
            engine.store.validate_evidence(r)
        quality_table = result_table(quality)
        display(quality_table[['task','method','complete_evidence_hit','complete_candidate_hit',
                               'best_group_coverage','evidence_count','jev_input_tokens','laya_input_tokens',
                               'retrieval_api_calls','api_cost_usd']])
        meta = runtime_manifest(ROOT,engine)
        meta.update({'experiment':EXPERIMENT_SNAPSHOT,'raw_log':RAW_SESSION_META,'mode':'cached_quality_abcd','split':SPLIT,
                     'truncation_audit':audit,'questions_file_sha256':file_sha256(ROOT/f'data/prepared/questions_{SPLIT}.jsonl')})
        quality_directory = save_run(ROOT,quality,meta,'05_measure_abcd.ipynb')
        print('Quality saved:',quality_directory)


### 일반 비교 · 실제 검색 시간·입력량·비용

네 경로는 각 질문을 독립적으로 검색하고 순서를 무작위로 섞는다. A/B는 로컬 GPU, C는 원격 Jev API, D는 로컬 Laya 모델이다. C와 D는 동일한 Jev 준비 트리와 원문을 쓰며 판단 비용·지연만 온라인에서 달라진다. D local inference 시간과 입력 tokens는 Jev API tokens와 분리해 기록한다. 완료한 질문별 원시 결과를 즉시 저장한다.


In [ ]:
if WORKFLOW == 'standard':
    online = []
    online_directory = None
    if engine:
        partial_directory = ROOT/'results/partial'/uuid4().hex
        partial_directory.mkdir(parents=True,exist_ok=False)
        meta = runtime_manifest(ROOT,engine)
        meta.update({'experiment':EXPERIMENT_SNAPSHOT,'raw_log':RAW_SESSION_META,'mode':'online_retrieval_abcd','split':SPLIT,'repeats':REPEATS,'truncation_audit':audit,
                     'notebook_source_sha256':notebook_source_sha256(ROOT/'notebook/05_measure_abcd.ipynb'),
                     'questions_file_sha256':file_sha256(ROOT/f'data/prepared/questions_{SPLIT}.jsonl')})
        write_json(partial_directory/'manifest.json',meta)
        def save_retrieval_record(record):
            with (partial_directory/'retrieval.jsonl').open('a',encoding='utf-8') as stream:
                stream.write(json.dumps(record,ensure_ascii=False)+'\n')
        online = measure_retrieval(engine,questions,labels,repeats=REPEATS,seed=CONFIG['seed'],
                                  methods=METHODS,on_result=save_retrieval_record,audit=RAW_LOG)
        online_directory = save_run(ROOT,online,meta,'05_measure_abcd.ipynb')
        write_json(partial_directory/'completed.json',{'run_directory':str(online_directory)})
        online_table = result_table(online)
        display(online_table[['task','method','retrieval_total_ms','complete_evidence_hit','evidence_count',
                              'jev_input_tokens','laya_input_tokens','laya_decision_calls','laya_truncated_calls',
                              'retrieval_api_calls','api_cost_usd','decision_stop_reason']])
        print('Online saved:',online_directory)


In [ ]:
if WORKFLOW == 'standard':
    if online:
        throughput = []
        for method,group in online_table.groupby('method'):
            row = {'method':method,'queries_per_wall_second':len(group)/(group.retrieval_total_ms.sum()/1000)}
            if method in ('A','B'):
                row['query_tokens_per_stage_second'] = group.retriever_input_tokens.sum()/(group.query_encode_ms.sum()/1000)
            if method=='B':
                row['rerank_pairs_per_stage_second'] = len(group)*min(RETRIEVE_K,corpus_size)/(group.rerank_ms.sum()/1000)
                row['rerank_tokens_per_stage_second'] = group.reranker_input_tokens.sum()/(group.rerank_ms.sum()/1000)
            if method=='C':
                request_seconds = sum(sum(c['seconds'] for c in r['trace']['calls']) for r in online if r['method']=='C')
                row['jev_input_tokens_per_request_second'] = group.jev_input_tokens.sum()/request_seconds
                row['jev_requests_per_request_second'] = group.retrieval_api_calls.sum()/request_seconds
            if method=='D':
                inference_seconds = group.laya_inference_seconds.sum()
                row['laya_input_tokens_per_inference_second'] = group.laya_input_tokens.sum()/inference_seconds
                row['laya_decision_calls_per_inference_second'] = group.laya_decision_calls.sum()/inference_seconds
            throughput.append(row)
        throughput_table = pd.DataFrame(throughput)
        throughput_table.to_csv(online_directory/'model_throughput.csv',index=False)
        display(throughput_table)
        prompt_previews = [{**build_prompt(r,encoding_name=GENERATION['encoding_name']),
                           'question_id':r['question_id'],'method':r['method']} for r in online]
        assert all(p['evidence_text_tokens']<=2000 for p in prompt_previews)
        write_jsonl(online_directory/'prompt_previews.jsonl',prompt_previews)
        write_json(online_directory/'truncation_audit.json',audit)
        display(pd.DataFrame([{'question_id':p['question_id'],'method':p['method'],
                              'evidence_text_tokens':p['evidence_text_tokens'],'passage_ids':p['passage_ids']} for p in prompt_previews]))
        for method in ('C','D'):
            example = next(r for r in online if r['method']==method)
            display(pd.DataFrame([{'method':method,'id':p['id'],'title':p['title'],'original_text':p['content']}
                                  for p in example['evidence']]))


### 일반 비교 · 공통 GPT-6 Luna 전체 시스템 측정

새 검색 → 같은 프롬프트 → 같은 LLM API까지 측정한다. 정답 EM/F1, 실제 생성 토큰, 검색 API 비용과 합산 API 비용을 저장한다. C는 원격 Jev 판단 요금, D는 로컬 Laya 추론을 사용한다. D의 API 비용 0은 GPU 전기·하드웨어 비용이 빠진 값이다.

LLM 호출 상한은 선택 문항 × 4방법 × 반복 수다. 개발 각 20·1회이면 160회, 평가 각 50·1회이면 400회다. 품질 평가와 독립 검색 지연 측정에도 C/D 판단 호출이 발생한다. 자동 재시도는 없고 완료 결과는 질문 단위로 저장한다.


In [ ]:
if WORKFLOW == 'standard':
    system_results = []
    system_directory = None
    if RUN_GENERATION and engine:
        from openai import OpenAI, DefaultHttpxClient
        if not os.environ.get('OPENAI_API_KEY'):
            raise RuntimeError('루트 .env에 OPENAI_API_KEY가 필요합니다.')
        system_partial_directory = ROOT/'results/partial'/uuid4().hex
        system_partial_directory.mkdir(parents=True,exist_ok=False)
        meta = runtime_manifest(ROOT,engine)
        meta.update({'experiment':EXPERIMENT_SNAPSHOT,'raw_log':RAW_SESSION_META,'mode':'live_system_abcd','generation':GENERATION,'split':SPLIT,'repeats':REPEATS,
                     'notebook_source_sha256':notebook_source_sha256(ROOT/'notebook/05_measure_abcd.ipynb'),
                     'questions_file_sha256':file_sha256(ROOT/f'data/prepared/questions_{SPLIT}.jsonl')})
        write_json(system_partial_directory/'manifest.json',meta)
        def save_system_record(record):
            with (system_partial_directory/'system.jsonl').open('a',encoding='utf-8') as stream:
                stream.write(json.dumps(record,ensure_ascii=False)+'\n')
        try:
            with DefaultHttpxClient(timeout=180,event_hooks=RAW_LOG.http_hooks if RAW_LOG else {}) as llm_http, \
                 OpenAI(max_retries=0,timeout=180,http_client=llm_http) as generator:
                system_results = measure_system(engine,questions,labels,client=generator,
                    generation_config=GENERATION,max_calls=MAX_LLM_API_CALLS,repeats=REPEATS,methods=METHODS,
                    seed=CONFIG['seed'],on_result=save_system_record,audit=RAW_LOG)
        finally:
            jev_client.close()
            if jev_http: jev_http.close()
        system_directory = save_run(ROOT,[r['retrieval'] for r in system_results],meta,'05_measure_abcd.ipynb')
        write_jsonl(system_directory/'system.jsonl',system_results)
        write_json(system_partial_directory/'completed.json',{'run_directory':str(system_directory)})
        print('System saved:',system_directory)
    elif jev_client:
        jev_client.close()
        if jev_http: jev_http.close()
        print('Generation is disabled; retrieval and quality results have been saved.')


In [ ]:
if WORKFLOW == 'standard':
    if system_results:
        rows = []
        for r in system_results:
            retrieved,generated = r['retrieval'],r['generation']
            usage = generated['api_usage'] or {}
            local = retrieved.get('local_usage') or {}
            rows.append({'question_id':retrieved['question_id'],'task':retrieved['task'],'method':retrieved['method'],
                         'repeat':retrieved['repeat'],'online_total_ms':r['online_total_seconds']*1000,
                         'diagnostic_capture_ms':r.get('diagnostic_capture_seconds',0)*1000,
                         'raw_events_path':(r.get('raw_log') or {}).get('events_path'),
                         'answer':generated['answer'],'valid_output':generated['valid_output'],
                         'evidence_count':len(retrieved['evidence']),
                         'complete_evidence_hit':retrieved['metrics']['complete_evidence_hit'],
                         'jev_input_tokens':(retrieved.get('api_usage') or {}).get('input_tokens'),
                         'laya_input_tokens':(retrieved.get('model_tokens',{}).get('laya') or {}).get('nonpad'),
                         'laya_decision_calls':local.get('calls'),'laya_truncated_calls':local.get('truncated_calls'),
                         'llm_input_tokens':usage.get('input_tokens'),'llm_output_tokens':usage.get('output_tokens'),
                         'retrieval_api_cost_usd':r['retrieval_api_cost_usd'],
                         'generation_api_cost_usd':r['generation_api_cost_usd'],
                         'online_api_cost_usd':r['online_api_cost_usd'],**r['answer_metrics']})
        system_table = pd.DataFrame(rows)
        system_table.to_csv(system_directory/'system.csv',index=False)
        display(system_table)
        system_summary = system_table.groupby(['task','method']).agg(
            questions=('question_id','nunique'),answer_em=('answer_em','mean'),answer_f1=('answer_f1','mean'),
            p50_ms=('online_total_ms','median'),api_cost_usd=('online_api_cost_usd',lambda x:x.sum(min_count=1)))
        system_summary.to_csv(system_directory/'system_summary.csv')
        display(system_summary)
        print('Measured split:',SPLIT,'| questions per task:',QUESTIONS_PER_TASK,'| repetitions:',REPEATS)
    else:
        print('No measured LLM API results.')
    if RAW_LOG:
        RAW_LOG.event('session','notebook_finished',{'quality_directory':str(quality_directory) if quality_directory else None,
            'online_directory':str(online_directory) if online_directory else None,
            'system_directory':str(system_directory) if system_directory else None})
        print('Question/method/phase/repeat logs:',RAW_LOG.directory)


### 일반 비교 · 해석 원칙과 다음 실험

D는 `config/d_laya.json`의 분기·완결성 임계값 0.5, 원문 Score 2.0, 최대 32회 로컬 판단을 임시로 사용한다. 이 값은 Jev 값과 등가라고 가정하지 않았다. 개발 데이터에서 임계값을 보정·고정한 뒤 평가셋을 실행해야 한다. Laya가 reported confidence를 제공하더라도 분기 선택은 현재의 `noul` 확률을 사용하며 잘림·중단·방문·원문 Score를 trace에 남긴다.

비교 해석은 두 층으로 나눈다. 온라인 검색은 같은 준비 트리의 Jev API 판단(C)과 로컬 Laya 판단(D)을 비교한다. 그 트리 생성 비용은 C/D 공통 준비 비용으로 한 번만 보고하며, D만의 독립적인 자료 분류 성능을 검증한 결과라고 부르지 않는다. 온라인 API 금액과 로컬 GPU 처리량·메모리는 별도로 해석한다.

다음 단계는 개발셋에서 D 정책을 정하고 고정한 뒤 더 큰 문항·corpus 설정을 비교하는 것이다. D의 한국어·긴 입력 처리, 잘림 없는 적용 범위와 근거 누락은 결과로 확인할 가설이다. Jev 공식 계약은 [API](https://docs.typesafe.ai/api), [고정 모델](https://docs.typesafe.ai/models)이다. Laya의 [공식 저장소/API](https://github.com/NandhaKishorM/laya), [모델 카드](https://huggingface.co/convaiinnovations/laya-multilingual)를 참고한다.
